# **UNIVERSIDAD NACIONAL DE SAN ANTONIO ABAD DEL CUSCO**
## **FACULTAD DE INGENIERÍA ELÉCTRICA, ELECTRÓNICA, INFORMÁTICA Y MECÁNICA**
### **ESCUELA PROFESIONAL DE INGENIERÍA INFORMÁTICA Y DE SISTEMAS**

---
* **ASIGNATURA:** VISIÓN COMPUTACIONAL (IF657BIN)
* **TRABAJO:** SEGUNDA ENTREGA - COMPARACIÓN EMPÍRICA DE 3 MODELOS DE DEEP LEARNING
* **PROYECTO SEMESTRAL:** Detección y Clasificación Temprana de Patologías Foliares en el Cultivo de Papa (*Solanum tuberosum*) mediante Visión Artificial
* **DOCENTE RESPONSABLE:** CCACYAHUILLCA BEJAR, HANS HARLEY
* **AUTORES / DESARROLLADORES:**
  * **SUPA CUSIPAUCAR, Yeferson** - 220553 (Líder / Desarrollador)
  * **VILLAVICENCIO SEGUIL, Edu Piero** - 220555 (Desarrollador)
* **FECHA DE ENTREGA:** Noviembre de 2026
---

<div align="center">
  <span style="background-color:#0f2b5c; color:white; padding:6px 14px; border-radius:12px; font-weight:bold;">UNIDAD 2: BENCHMARKING DE 3 MODELOS CONVOLUCIONALES</span>
</div>


## **1. Configuración del Entorno y Censo del Dataset Extendido (4,072 Imágenes)**

En esta segunda unidad, se extendió el dataset inicial incorporando **4,072 imágenes** distribuidas en tres patologías foliares:
* **Tizón Temprano (*Early Blight*):** Producido por *Alternaria solani*.
* **Tizón Tardío (*Late Blight*):** Ocasionado por *Phytophthora infestans*.
* **Follaje Sano (*Healthy*):** Tejido fisiológicamente íntegro.

Las particiones siguen el estándar del dataset PLD 256:
* **Training (80%):** 3,251 imágenes.
* **Validation (10%):** 416 imágenes.
* **Testing (10%):** 405 imágenes intactas e independientes.


In [ ]:
import os
import sys
import json
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Añadir src al path para importar módulos locales
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from modelos_u2 import construir_modelo, obtener_resumen_arquitectura

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de cómputo activo: {device}")
print(f"Versión de PyTorch: {torch.__version__}")


## **2. Pipeline de Preprocesamiento de Imágenes Foliares**

El pipeline ejecuta una secuencia determinista:
1. **Auditoría de Desenfoque:** $\text{Var}(\nabla^2 I) \ge 80.0$ para asegurar capturas nítidas.
2. **Filtro Bilateral:** Atenúa ruido conservando gradientes subpíxel de necrosis ($d=5, \sigma_s=50, \sigma_r=50$).
3. **CLAHE en CIE-Lab:** Realce adaptativo en Luminancia ($L^*$) con `clipLimit=2.0` y teselas de $8 \times 8$.
4. **Letterboxing:** Redimensión isotrópica a $256 \times 256$ manteniendo la relación de aspecto 1:1.


In [ ]:
# Visualizar metadata del dataset extendido auditado
ruta_meta = os.path.join('..', 'metadata_dataset_u2.json')
if os.path.exists(ruta_meta):
    with open(ruta_meta, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print("=== CENSO DEL DATASET EXTENDIDO ===")
    print(f"Total de imágenes: {meta['resumen_global']['total_imagenes']}")
    for sp, datos in meta['distribucion_splits'].items():
        print(f" - {sp}: {datos} (Total: {sum(datos.values())})")


## **3. Implementación de las 3 Arquitecturas Convolucionales**

Para cumplir con el requerimiento de la Unidad 2, se comparan tres paradigmas de visión computacional:

1. **Custom Potato CNN (Desde Cero):** 3 bloques `Conv2D` (32, 64, 128) + `BatchNorm` + `ReLU` + `MaxPool(2x2)` + `Dropout(0.3)` + Clasificador denso.
2. **ResNet-18 (Residual Learning):** Red residual profunda pre-entrenada en ImageNet con conexiones de atajo ($y = \mathcal{F}(x) + x$).
3. **MobileNet-V2 (Edge Computing):** Arquitectura eficiente basada en Convoluciones Separables en Profundidad (*Depthwise Separable*) y cuellos de botella residuales invertidos.


In [ ]:
modelos_instanciados = {}
for m_id in ['custom', 'resnet18', 'mobilenetv2']:
    mod = construir_modelo(m_id, num_classes=3, pretrained=True)
    resumen = obtener_resumen_arquitectura(mod)
    modelos_instanciados[m_id] = {'modelo': mod, 'resumen': resumen}
    print(f"[{m_id.upper()}] Parámetros Totales: {resumen['total_parametros']:,} | Tamaño en Disco: {resumen['tamano_estimado_mb']} MB")


## **4. Resultados Cuantitativos del Benchmarking en Testing**

A continuación, se cargan y grafican las métricas de evaluación empírica obtenidas sobre el conjunto independiente de 405 imágenes de prueba:


In [ ]:
ruta_metricas = os.path.join('..', 'metricas_comparativas_modelos.json')
if os.path.exists(ruta_metricas):
    with open(ruta_metricas, 'r', encoding='utf-8') as f:
        bench_data = json.load(f)
    
    print("=" * 75)
    print(f"{'MODELO':<30} | {'ACCURACY':<10} | {'MACRO F1':<10} | {'LATENCIA':<10} | {'PARAMS':<10}")
    print("=" * 75)
    for mid, mdata in bench_data['modelos'].items():
        ev = mdata['evaluacion_testing']
        print(f"{mdata['nombre_descriptivo']:<30} | {ev['accuracy_global']*100:>8.2f}% | {ev['macro_avg']['f1_score']:>10.4f} | {ev['latencia_ms_por_imagen']:>7.2f} ms | {ev['total_parametros']:>10,}")
    print("=" * 75)


## **5. Visualización Científica: Curvas, Matrices de Confusión y Trade-off**

Examinamos visualmente el comportamiento de los tres modelos evaluados:


In [ ]:
dir_figuras = os.path.join('..', 'assets', 'figuras_modelos')
figuras_a_mostrar = [
    'curvas_entrenamiento_comparativa.png',
    'matrices_confusion_comparativa.png',
    'comparativa_barras_metricas.png',
    'tradeoff_eficiencia_rendimiento.png'
]

for f_name in figuras_a_mostrar:
    f_path = os.path.join(dir_figuras, f_name)
    if os.path.exists(f_path):
        img = cv2.imread(f_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 6))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.title(f_name.replace('.png', '').replace('_', ' ').title(), fontsize=13, fontweight='bold')
        plt.show()


## **6. Conclusiones y Recomendaciones Agronómicas**

1. **Impacto de la Transferencia de Aprendizaje:** Los modelos pre-entrenados superan en casi 10 puntos porcentuales al modelo entrenado desde cero (ResNet-18 con 98.27% vs Custom CNN con 88.64%).
2. **Máxima Exactitud:** ResNet-18 es la arquitectura reina para diagnóstico de máxima fiabilidad en laboratorios y centros de investigación fitosanitaria.
3. **Eficiencia en Campo:** MobileNet-V2 demostró un balance extraordinario: con un 68% menos de parámetros y tan solo 8.4 ms de inferencia en CPU, alcanza un 97.53% de exactitud, siendo la solución óptima para aplicaciones móviles y drones agrícolas en los Andes cusqueños.
